# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract answer 1 — what one row means for my lane

**In the warehouse, one row of `fact_content_daily_performance` = one page on one day for one client** (`report_date` × `client_hash_id` × `content_hash_id`). That is a *page-day*, not a page.

My lane (Lane 2, refresh scoring) scores **pages**, not page-days. So the contract has two grains and one deliberate step between them: I read page-days, then aggregate to **one row = one page** over a stated month. Query 1 proves the page-day grain actually holds before I aggregate anything on top of it.

### Contract answer 2 — which tables

- `fact_content_daily_performance`, partition `month=2026-03` — every feature comes from here.
- `fact_content_daily_performance`, partition `month=2026-04` — **label only**, never features.
- `dim_clients` — context on history start dates; not joined into the model frame.

I do not use `fact_content_query_90d`: its fixed 90-day window overlaps my March/April split, so joining it would drag April information into a March feature. Aligning that window is a bigger job than this contract needs.

### Contract answer 3 — which time window

| Window | Dates | Used for |
|---|---|---|
| Feature window | 2026-03-01 → 2026-03-31 | All five features |
| **Decision moment** | **2026-03-31** | Everything a feature uses must exist by this instant |
| Label window | 2026-04-01 → 2026-04-30 | The outcome only |

Splitting the windows is the point. "Knowable at the decision moment" means nothing until the decision moment is a specific date — here, the last day of March.

I use `month=2026-03` deliberately because it sits mid-panel. The `_sample` table is not a random sample but exactly the final month (June 2026), which is the natural outcome window for any forward-looking label, so I keep June sealed and never develop label logic on it.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Contract answer 4 — what I predict

**Label:** `is_declining_label = (April impressions < March impressions)`, computed per page.

This is a genuine forward-looking label, and it is the one real upgrade over the starter dataset. There, `trend_direction` arrived pre-bucketed, so I was predicting a *recorded trend bucket*. Here the outcome sits in a month my features cannot see, which is what makes "knowable at the decision moment" testable rather than decorative.

### Contract answer 5 — what I deliberately exclude

**Every April column, including `impressions_apr` itself.** April is the label window. Any April value is the answer in disguise, and Section 3 shows exactly what happens when one slips in.

### Field buckets

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `impressions_mar`, `ctr_mar`, `avg_position_mar`, `days_with_impressions`, `late_vs_early_ratio` | All aggregated from March only; every one exists by 2026-03-31 |
| **Label** | `impressions_apr` → `is_declining_label` | The outcome. Never a feature. |
| **Context** | `content_hash_id`, `client_hash_id`, `report_date`, `month` | Grouping, joining, splitting. `client_hash_id` is the split key so no client sits in both train and test. IDs are pseudonyms and never features. |
| **Excluded** | All GA4 columns (`ga4_*`, `sessions_*`, `scroll_events`) | Only 4.2% of March rows have `ga4_data_available IS TRUE` (Query 3). The rest are zero-filled, and a zero there means "not measured", not "no engagement". Section 4 has the numbers. |
| **Excluded** | `gsc_sum_position` as a raw feature | Kept only as the numerator for impression-weighted position. Raw, it is a function of impression volume and would double-count it. |
| **Excluded** | `fact_content_query_90d` entirely | Its 90-day window straddles the March/April boundary — a window-alignment leak. |


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
# Setup. The token is read from Colab Secrets, an environment variable, or a gitignored
# local .env -- never typed into a cell, because this repo is public.
import os
from pathlib import Path
import duckdb, pandas as pd
from huggingface_hub import hf_hub_download

def _read_env_file(path):
    """Parse KEY = value pairs, tolerating spaces, quotes and a BOM."""
    pairs = {}
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        pairs[key.strip()] = value.strip().strip('"').strip("'")
    return pairs

def get_token():
    try:                                    # Colab: key panel -> HF_TOKEN
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    if os.environ.get("HF_TOKEN"):          # already exported
        return os.environ["HF_TOKEN"]
    for parent in [Path.cwd(), *Path.cwd().parents]:   # local .env (gitignored)
        env = parent / ".env"
        if env.exists():
            token = _read_env_file(env).get("HF_TOKEN")
            if token:
                return token
    from getpass import getpass                        # last resort, hidden input
    return getpass("HF READ token: ")

TOKEN = get_token()
assert TOKEN, "No HF token found -- set HF_TOKEN or add it to .env"
print(f"token loaded: {TOKEN[:3]}... ({len(TOKEN)} chars)")   # never print it whole

DS = "FlyRank/internship-warehouse"
# Cache the two month partitions once. The data skill warns that repeated full scans
# hit HTTP 429 rate limits, so download-then-query beats re-scanning over the network.
MAR = hf_hub_download(DS, "fact_content_daily_performance/month=2026-03/data_0.parquet",
                      repo_type="dataset", token=TOKEN)
APR = hf_hub_download(DS, "fact_content_daily_performance/month=2026-04/data_0.parquet",
                      repo_type="dataset", token=TOKEN)
con = duckdb.connect()
print("March + April partitions ready.")


token loaded: hf_... (37 chars)


March + April partitions ready.


In [2]:
# QUERY 1 of 3 -- GRAIN. Claim: one row = one page-day (date x client x content).
# If that is true, no combination of the three appears twice. Zero rows back = grain holds.
q1 = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM '{MAR}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

print(f"duplicate page-days found: {len(q1)}")
print("GRAIN HOLDS -- one row really is one page-day." if len(q1) == 0
      else "GRAIN BROKEN -- do not aggregate until this is understood.")
q1


duplicate page-days found: 0
GRAIN HOLDS -- one row really is one page-day.


,report_date,client_hash_id,content_hash_id,n


In [3]:
# QUERY 2 of 3 -- ROW COUNT AND DATE SPAN of my slice.
# Claim: the partition covers exactly March 2026 and nothing else.
q2 = con.execute(f"""
    SELECT COUNT(*)                        AS page_days,
           COUNT(DISTINCT content_hash_id) AS pages,
           COUNT(DISTINCT client_hash_id)  AS clients,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day
    FROM '{MAR}'
""").fetchdf()
q2


,page_days,pages,clients,first_day,last_day
0,9841378,331437,55,2026-03-01,2026-03-31


In [4]:
# QUERY 3 of 3 -- AVAILABILITY, filtered with IS TRUE.
# A row existing does NOT mean it was measured. Rows before a client's data start are
# zero-FILLED with the availability flag FALSE, so a 0 there means "not measured",
# not "no traffic". IS TRUE is the only honest filter.
q3 = con.execute(f"""
    SELECT COUNT(*)                                                     AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)           AS gsc_available,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)           AS ga4_available,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                              AND ga4_data_available IS TRUE)           AS both_available
    FROM '{MAR}'
""").fetchdf()

r = q3.iloc[0]
print(f"rows surviving 'gsc_data_available IS TRUE' : {r.gsc_available:>10,}  "
      f"({r.gsc_available / r.all_rows:.1%} of {r.all_rows:,})")
print(f"rows surviving 'ga4_data_available IS TRUE' : {r.ga4_available:>10,}  "
      f"({r.ga4_available / r.all_rows:.1%})")
print(f"rows surviving BOTH flags                   : {r.both_available:>10,}  "
      f"({r.both_available / r.all_rows:.1%})")
print("\n-> GSC is usable. GA4 is not, at this coverage. That decides my feature list.")
q3


rows surviving 'gsc_data_available IS TRUE' :  3,611,061  (36.7% of 9,841,378)
rows surviving 'ga4_data_available IS TRUE' :    413,966  (4.2%)
rows surviving BOTH flags                   :    364,347  (3.7%)

-> GSC is usable. GA4 is not, at this coverage. That decides my feature list.


,all_rows,gsc_available,ga4_available,both_available
0,9841378,3611061,413966,364347


### Five features, max — and when each is available

Decision moment: **2026-03-31**. Every feature is aggregated from March rows only, and every one passes the same test: *could I have computed this on the evening of 31 March?*

| # | Feature | Knowable at the decision moment because… |
|---|---|---|
| 1 | `impressions_mar` | It sums GSC impressions dated 1–31 March. Every row it touches was already recorded by 31 March. |
| 2 | `ctr_mar` | Clicks ÷ impressions, both from March rows only. A ratio of two things already measured is knowable whenever they are. |
| 3 | `avg_position_mar` | `SUM(gsc_sum_position) / SUM(gsc_impressions)` over March — impression-weighted, so a page ranking well on its busy days is not outvoted by a quiet day. All inputs are March. |
| 4 | `days_with_impressions` | Counts March days with any impression. Bounded by the 31 days that had already happened. |
| 5 | `late_vs_early_ratio` | Impressions in 22–31 March ÷ impressions in 1–10 March. Both windows close *before* the decision moment, so this measures momentum inside March without touching April. |

Feature 5 is the one worth pausing on. It looks like it might be the label, because both describe a direction of travel — but it compares two windows *inside* March, while the label compares March to April. That is the whole distinction between a legitimate trend feature and a leak: not what it measures, but **when the data it reads became available**.

All five come from GSC. Query 3 is the reason: GA4 is present on only 4.2% of March rows, so an engagement feature would be missing for most pages, and filling those blanks with zero would inject "this client had no GA4 yet" as if it were "nobody engaged".


In [5]:
# Build the feature frame: page-days -> ONE ROW = ONE PAGE.
# Features from March only. Label from April, kept in its own CTE so the boundary is visible.
frame = con.execute(f"""
    WITH march AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions)                                     AS impressions_mar,
               SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)        AS ctr_mar,
               SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0)  AS avg_position_mar,
               COUNT(*) FILTER (WHERE gsc_impressions > 0)              AS days_with_impressions,
               SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-22')
                 / NULLIF(SUM(gsc_impressions)
                          FILTER (WHERE report_date <= DATE '2026-03-10'), 0)
                                                                        AS late_vs_early_ratio
        FROM '{MAR}'
        WHERE gsc_data_available IS TRUE          -- availability, not mere row existence
        GROUP BY 1, 2
    ),
    april AS (                                     -- LABEL WINDOW. Never a feature source.
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_apr
        FROM '{APR}'
        WHERE gsc_data_available IS TRUE
        GROUP BY 1
    )
    SELECT march.*,
           COALESCE(april.impressions_apr, 0) AS impressions_apr,
           CASE WHEN COALESCE(april.impressions_apr, 0) < march.impressions_mar
                THEN 1 ELSE 0 END              AS is_declining_label
    FROM march LEFT JOIN april USING (content_hash_id)
    WHERE march.impressions_mar >= 100         -- minimum volume, or the ratios are noise
""").fetchdf()

FEATURES = ["impressions_mar", "ctr_mar", "avg_position_mar",
            "days_with_impressions", "late_vs_early_ratio"]

print(f"one row = one page | rows: {len(frame):,} | clients: {frame.client_hash_id.nunique()}")
print(f"label rate (declining): {frame.is_declining_label.mean():.3f}\n")
print("missing values per feature:")
print(frame[FEATURES].isna().sum().to_string())
print("\n-> late_vs_early_ratio is null when a page had zero impressions in 1-10 March,")
print("   i.e. the denominator did not exist. That is 'not computable', not 'no change'.")
frame.head(5)


one row = one page | rows: 101,441 | clients: 44
label rate (declining): 0.658

missing values per feature:
impressions_mar             0
ctr_mar                     0
avg_position_mar            0
days_with_impressions       0
late_vs_early_ratio      9387

-> late_vs_early_ratio is null when a page had zero impressions in 1-10 March,
   i.e. the denominator did not exist. That is 'not computable', not 'no change'.


,content_hash_id,client_hash_id,impressions_mar,ctr_mar,avg_position_mar,days_with_impressions,late_vs_early_ratio,impressions_apr,is_declining_label
0,content_76a2cf588a7b9756,client_73cda7b4e4f265ea,264.0,0.000000,9.609848,31,0.898990,231.0,1
1,content_dbfe83f2e9789f95,client_73cda7b4e4f265ea,114.0,0.000000,7.236842,29,0.888889,83.0,1
2,content_6b9f2a9115af232f,client_73cda7b4e4f265ea,758.0,0.001319,6.406332,31,0.664634,1077.0,0
3,content_b507bf3ee085fe6e,client_73cda7b4e4f265ea,654.0,0.001529,48.824159,31,0.340517,225.0,1
4,content_e3f58ebbe68ecbad,client_73cda7b4e4f265ea,2145.0,0.004196,4.560373,31,0.799733,2403.0,0


### The trap — one label-derived column, on purpose

Now I break the contract deliberately. I add **one** column, `impressions_delta = impressions_apr − impressions_mar`, and score the same model twice.

`impressions_delta` is exactly what the label is computed from: `is_declining_label` is just `impressions_delta < 0`. It is the warehouse version of `trend_pct` in notebook 02 — a feature that *is* the answer, wearing a different name.

The point of running it rather than reading about it: a leak does not announce itself with an error. It announces itself as **a very good score**, which is the one result nobody feels like interrogating.


In [6]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

d = frame.replace([np.inf, -np.inf], np.nan)
d["late_vs_early_ratio"] = d["late_vs_early_ratio"].fillna(1.0)  # 1.0 = "no measurable change"
y = d["is_declining_label"].values

# Client-grouped split: no client's pages in both train and test.
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
              .split(d, y, groups=d["client_hash_id"]))

def quick_score(cols):
    m = DecisionTreeClassifier(max_depth=3, random_state=42).fit(d.iloc[tr][cols], y[tr])
    return roc_auc_score(y[te], m.predict_proba(d.iloc[te][cols])[:, 1])

honest = quick_score(FEATURES)

# --- BREAK THE CONTRACT ON PURPOSE -------------------------------------------
d["impressions_delta"] = d["impressions_apr"] - d["impressions_mar"]
leaked = quick_score(FEATURES + ["impressions_delta"])

print(f"train clients: {d.iloc[tr].client_hash_id.nunique()}  |  "
      f"test clients: {d.iloc[te].client_hash_id.nunique()}\n")
print(f"honest  (5 contract features)  ROC-AUC = {honest:.3f}")
print(f"LEAKED  (+ impressions_delta)  ROC-AUC = {leaked:.3f}   <- 'perfect', and worthless")

# --- DELETE IT AND KEEP THE HONEST NUMBER ------------------------------------
d = d.drop(columns=["impressions_delta"])
assert "impressions_delta" not in d.columns
assert not any(c.endswith("_apr") for c in FEATURES)
print(f"\nleak column removed. Columns now: {sorted(d.columns.difference(['content_hash_id','client_hash_id']))}")
print(f"THE NUMBER I KEEP AND REPORT: ROC-AUC = {honest:.3f}")


train clients: 33  |  test clients: 11

honest  (5 contract features)  ROC-AUC = 0.682
LEAKED  (+ impressions_delta)  ROC-AUC = 1.000   <- 'perfect', and worthless

leak column removed. Columns now: ['avg_position_mar', 'ctr_mar', 'days_with_impressions', 'impressions_apr', 'impressions_mar', 'is_declining_label', 'late_vs_early_ratio']
THE NUMBER I KEEP AND REPORT: ROC-AUC = 0.682


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### The named limitation: my slice is GSC-only, and that is not a choice I made

Query 3 measured it. Of **9,841,378** March page-days:

| Filter | Rows surviving | Share |
|---|---|---|
| `gsc_data_available IS TRUE` | 3,611,061 | 36.7% |
| `ga4_data_available IS TRUE` | 413,966 | **4.2%** |
| both | 364,347 | 3.7% |

**What this means my analysis can never tell you.** Nothing about on-page behaviour. Whether a declining page still *held* the readers it got — engagement rate, scroll depth, session quality — is unanswerable here, because GA4 exists for 4.2% of rows. Everything I can say is about how a page appears in search results, not what happens after the click.

That distinction matters for the lane. "Refresh this page" assumes the content is underperforming, but a page can lose impressions while serving its remaining readers perfectly well. My features cannot separate those two cases.

**Why the zero-fill makes it worse than a normal missing value.** Rows before a client's `ga4_data_start` are not blank — they are filled with zeros and flagged `ga4_data_available = FALSE`. A blank announces itself; a zero looks like a measurement. Any average over unfiltered rows silently mixes "nobody engaged" with "we were not recording yet", and it will be wrong in the direction that flatters the analysis.

**Two more limits worth stating plainly.** March holds 55 of the warehouse's 104 clients, so roughly half have no March history at all and my slice is not a cross-section of FlyRank's clients — it is a cross-section of clients *already onboarded by March 2026*. And the whole result rests on one month-pair; a March→April pattern is one observation of one seasonal period, not a stable law.

All of this is **observed and directional**: measured on one slice, useful for deciding where to look next, and not evidence about what causes a page to decline.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only hashed IDs, and the token is never printed or pasted into a cell
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
